In [ ]:
# Sirket_Harcama_Kar_Bilgileri.csv 
# veriseti ?oklu regresyon

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/erkansirin78/datasets/master/Sirket_Harcama_Kar_Bilgileri.csv")

# 1. Veri Ke?fi

In [ ]:
df.head()

## 1.1. Veri seti ?emas?n?, bo? de?er say?s?n? g?rme

In [ ]:
df.info()

## 1.2. Histogram

In [ ]:
df.hist(bins=15, figsize=(20,15))
plt.show()

## 1.3. Korelasyonlar

In [ ]:
df.corr()

In [ ]:
# Girdi de?i?kenlerin hedef de?i?ken ile korelasyonlar?
df.corrwith(df['Kar']).sort_values(ascending=False)

## 1.4. N?merik nitelikleri inceleme

In [ ]:
df.describe().T

## 1.5. S?tunlar? yeniden s?ralama

In [ ]:
# ?stedi?im gibi de?i?tirmiyorum

# 2. Veri Temizli?i ve ?n Haz?rl???

## 2.1. Kay?p veri var m??

In [ ]:
df.isnull().sum()

## 2.2. SimpleImputer ile total_bedrooms niteli?indeki bo?lar? doldurma

In [ ]:
# Bo? de?erim yok

In [ ]:
df.head()

## 2.3. Veriyi X,y olarak b?lme (nitelikler matrisi ve hedef de?i?ken - dikey b?lme) aka Numpy D?nyas?na Giri? 

In [ ]:
# Hedef de?i?ken art?k en sonda
X = df.iloc[:,:-1].values
X.shape

In [ ]:
y = df.iloc[:,-1].values
y.shape

## 2.4. LabelEncoder ile kategorik nitelikleri kodlama

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder_sehir = LabelEncoder()

In [ ]:
X[:3,:]

In [ ]:
X[:,3] = label_encoder_sehir.fit_transform(X[:,3])
X.shape

In [ ]:
X[:3,:]

## 2.5. OneHotEncoder ile kodlanan kategorik nitelikleri g?lge de?i?kenler yapma

In [ ]:
from sklearn.preprocessing import OneHotEncoder
onehotencoder = OneHotEncoder(handle_unknown='ignore', categories='auto')

In [ ]:
# Tek bir kategorik nitalik var. Acaba ka? farkl? kategori var?
len(df.Sehir.unique())

In [ ]:
# Bir veya daha fazla s?tunu ilgilendiren d?n???mleri kolay yapabilmek i?in kullan??l? bir s?n?f (Estimator k?kenli)
# Biz burada sadece OneHotEncoder i?in kullanaca??z.
from sklearn.compose import ColumnTransformer

In [ ]:
# ct bizim kendi verdi?iiz isimdir. 
# onehotencoder'? yukar?da yaratt?k. [0] ise d?n??ecek niteliklerin indeks de?erleridir.

ct = ColumnTransformer(
    [('one_hot_encode_with_ct', onehotencoder, [3]),], 
    remainder='passthrough'
)

In [ ]:
# ColumnTransformer nesnesi ile onehotencoder nesnesini e?itip gerekli d?n???mleri ger?ekle?tirip X'e tekrar atayal?m.
# Burada X'in boyutu(shape) de?i?ecektir.
X = ct.fit_transform(X)

In [ ]:
X.shape

In [ ]:
# Toplam 4 niteli?imiz vard? 3 farkl? ?ehir i?in g?lge de?i?ken olu?tu 7. 
# LabelEncoder ile olu?an nitelik d??t? kald? 6. Hesap do?ru

In [ ]:
X[:3,:]

## 2.6. G?lge De?i?ken Tuza??ndan (Dummy variable trap) Ka??nma 

In [ ]:
X = X[:,0:]
X.shape

## 2.7. Hedef de?i?ken i?in Label Encoder

In [ ]:
# Hedef niteli?i zaten rakam oldu?u i?in kodlamaya gerek yok. Ayr?ca problem regresyon problemi

## 2.8. Standardizasyon FeatureScaling

In [ ]:
# Harcamalar ayn? t?rden yapmayabiliriz.

## 2.9. Veri setini train-test olarak ay?rma (yatay b?lme)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,
            test_size=0.25, random_state = 142)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# 3. Model Olu?turma ve E?itim

## 3.1. Lineer Regresyon Nesnesi Olu?turma (Bo? model yaratma)

In [ ]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()

## 3.2. Lineer modeli e?itme (modeli ?ekillendirme - veriye uydurma - e?itme)

In [ ]:
regressor.fit(X_train, y_train)

## 3.3. E?itilmi? modelle (makine) test seti kullanarak tahmin (prediction) 

In [ ]:
y_pred = regressor.predict(X_test)

## 3.4. Tahmin sonu?lar?n? numpy.ndarray'den dataframe yapma (?ncelemek daha Kolay)

In [ ]:
predictions = pd.DataFrame(data=y_pred,    # values
                index=range(len(y_pred)),    # 1st column as index
                   columns=['y_pred'])  # 1st row as the column names

# Sadece y_pred'den olu?an df'e test(ger?ek) y_test'i s?tun olarak ekleme
predictions['y_test'] = y_test
predictions.head()

## 3.5. Regression metrics

### 3.5.1. explained_variance

In [ ]:
# The sklearn.metrics module implements several loss, score, and utility functions to measure 
# regression performance. Some of those have been enhanced to handle the multioutput case: 
# mean_squared_error, mean_absolute_error, explained_variance_score and r2_score.

In [ ]:
from sklearn.metrics import explained_variance_score

In [ ]:
explained_variance_score(y_true=y_test, y_pred=y_pred)

### 3.5.2. mean_absolute_error

In [ ]:
from sklearn.metrics import mean_absolute_error

In [ ]:
mean_absolute_error(y_pred=y_pred, y_true=y_test)

In [ ]:
import math

In [ ]:
# rmse
math.sqrt(mean_absolute_error(y_pred=y_pred, y_true=y_test))

### 3.5.3. r2_score 

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(y_test, y_pred)

# 4. Model Anlaml?l???

In [ ]:
# Model anlaml?l???na ve girdi de?i?kenlerin modele katk?s?na bakmak i?in scikit-learn'den farkl?
# bir k?t?phane kullanaca??z. ??nk? bunu scikit learn'da yapam?yoruz. 

In [ ]:
import statsmodels.api as sm

## 4.1. Sabit ekleme

In [ ]:
# Sabit Ekleme (asl?nda sbitin ?n?nde bir tane x ?zeri 0 var. 
# Onu temsilen yeni bir s?tunu 1'ler ile dolduruyoruz)

In [ ]:
X.shape

In [ ]:
X = sm.add_constant(X)
X.shape

In [ ]:
X[:5,:]

    Hatay? engellemek i?in a?a??daki i?lemi yap?yoruz.
    TypeError: ufunc 'isfinite' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
X_opt = np.array(X, dtype=float)

In [ ]:
import statsmodels.formula.api as sm
ols_obj = sm.OLS(endog=y, exog=X_opt)

In [ ]:
ols_model = ols_obj.fit()

In [ ]:
ols_model.summary()

In [ ]:
df.head()

# 5. Model Olu?turma (Geriye Do?ru Eleme Y?ntemi)

## 1. Tur

In [ ]:
# Sabit -> X[0]
# x1 : ?ehir-1 -> X[1]
# x2 : ?ehir-2 -> X[2]
# x3 : ArgeHarcamasi -> X[3]
# x4 : YonetimGiderleri -> X[4]
# x5 : PazarlamaHarcamasi -> X[5]

In [ ]:
# Demekki ?ehir niteli?i kar? a??klamada anlams?z ben bu niteli?i modele hi? katm?yorum.
# Modele girecekler 0: Sabit, 3: Arge, 4: Y?netim, 5: Pazarlama

In [ ]:
import statsmodels.formula.api as sm
X_opt = np.array(X[:, [0, 3,4,5]], dtype=float)
ols_obj = sm.OLS(endog=y, exog=X_opt)

In [ ]:
ols_model = ols_obj.fit()

In [ ]:
ols_model.summary()

# 2. Tur

In [ ]:
# Sabit -> X[0]
# x1 : ArgeHarcamasi -> X[3]
# x2 : YonetimGiderleri -> X[4]
# x3 : PazarlamaHarcamasi -> X[5]

In [ ]:
# YonetimGiderleri niteli?i kar? a??klamada anlams?z bu niteli?i modelden ??kar?yoruz.
# Modele girecekler 0: Sabit, 3: Arge, 5: Pazarlama

In [ ]:
import statsmodels.formula.api as sm
X_opt = np.array(X[:, [0, 3, 5]], dtype=float)
ols_obj = sm.OLS(endog=y, exog=X_opt)

In [ ]:
ols_model = ols_obj.fit()

In [ ]:
ols_model.summary()

In [ ]:
# Sabit -> X[0]
# x1 : ArgeHarcamasi -> X[3]
# x2 : PazarlamaHarcamasi -> X[5]

In [ ]:
# Pazarlama harcamas? ?ok az bir farkla anlams?z kal?yor o.06 > 0.05
# ?stersek bunu da ??karabiliriz veya kalabilir.